In [3]:
import sqlite3
import pandas as pd 

conn = sqlite3.connect("../data/olist.db")

def q(sql):
    return pd.read_sql(sql, conn)

# Блок 1. Продажи и категории

Выручка = сумма price по заказам со статусом delivered (без стоимости доставки freight_value). Клиент - уникальный customer_unique_id , а не customer_id. customer_id создаётся заново для каждого заказа.  

In [4]:
q("""
SELECT COUNT(*) AS orders_count,
       COUNT(DISTINCT customer_id) AS customer_ids,
       (SELECT COUNT(DISTINCT customer_unique_id) FROM customers) AS unique_customers_count
FROM orders;
""")

,orders_count,customer_ids,unique_customers_count
0,99441,99441,96096


In [6]:
q("""
SELECT MIN(order_purchase_timestamp) AS first_order,
       MAX(order_purchase_timestamp) AS last_order
FROM orders;
""")

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [ ]:
q("""
SELECT order_status,
       COUNT(*) AS orders_count,
       ROUND(COUNT(*)/(SELECT COUNT(*) FROM orders), 2) as percent
FROM orders
GROUP BY order_status
ORDER BY orders_count DESC;
""")

Подавляющее большинство заказов в статусе delivered.
Далее весь анализ делается по доставленным заказам

In [ ]:
q("""
SELECT strftime('%Y-%m', orders.order_purchase_timestamp) AS month,
       COUNT(DISTINCT orders.order_id) AS orders,
       ROUND(SUM(order_items.price),2) AS revenue
FROM orders
JOIN order_items ON orders.order_id = order_items.order_id
WHERE orders.order_status = 'delivered'
GROUP BY month
ORDER BY month;
""")

Наибольшая выручка в ноябре 2017. Месяцы с малым числом заказов (конец 2016 года и сентябрь-октябрь 2018 года) неполные

In [ ]:
q("""
SELECT payment_type,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(SUM(payment_value),2) AS total_paid
FROM order_payments
GROUP BY payment_type
ORDER BY total_paid DESC;
""")

In [ ]:
q("""
SELECT ROUND(SUM(oi.price)/COUNT(DISTINCT oi.order_id)) AS avg_order_value
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered';
""")

Средний чек = 137

Какие ТОП-10 категорий по выручке?

In [ ]:
q("""
SELECT COALESCE(t.product_category_name, t.product_category_name_english) AS category,
       COUNT(DISTINCT oi.order_id) AS orders,
       ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation t 
       ON p.product_category_name = t.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;
""")

Какие ТОП-10 штатов клиентов по выручке?

In [ ]:
q("""
SELECT c.customer_state AS state,
       COUNT(DISTINCT oi.order_id) AS orders,
       ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY state
ORDER BY revenue DESC
LIMIT 10;
""")

# Блок 2. Доставка и отзывы

In [ ]:
q("""
SELECT ROUND(AVG(julianday(order_delivered_customer_date)-
               julianday(order_purchase_timestamp)), 1) AS avg_days
FROM orders
WHERE order_status = 'delivered' 
  AND order_delivered_customer_date IS NOT NULL;
""")

In [17]:
q("""
SELECT COUNT(*) AS orders,
       SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1 ELSE 0 END) AS late_orders,
       ROUND(100*SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date
                          THEN 1 ELSE 0 END)/COUNT(*), 2) AS late_percent
FROM orders
WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL;
""")

,orders,late_orders,late_percent
0,96470,7826,8.0


In [ ]:
q("""
SELECT review_score,
       COUNT(*) AS reviews,
       ROUND(COUNT(*)*100/(SELECT COUNT(*) FROM order_reviews), 2) AS percent
FROM order_reviews
GROUP BY review_score
ORDER BY review_score;
""")

In [ ]:
q("""
SELECT CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 'late' ELSE 'on_time' END AS delivery,
       COUNT(*) AS reviews,
       ROUND(AVG(r.review_score),2) AS avg_score
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
GROUP BY delivery;
""")


У опоздавших заказов средняя оценка ниже: 2.57, в то время как у доставленных вовремя 4.29

In [ ]:
q("""
SELECT CASE 
         WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <=0 THEN '0. Вовремя'
         WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <=3 THEN '1. До 3 дней'
         WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <=7 THEN '2. До 7 дней'
         ELSE '3. Больше 7 дней'
       END AS delay_bucket,
       COUNT(*) AS reviews,
       ROUND(AVG(r.review_score), 2) AS avg_score
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
GROUP BY delay_bucket
ORDER BY delay_bucket;
""")

Видно, что оценка падает по мере роста опоздания

In [ ]:
q("""
SELECT c.customer_state,
       COUNT(*) AS orders,
       ROUND(AVG(julianday(o.order_delivered_customer_date)
                - julianday(o.order_purchase_timestamp)), 1) AS avg_days,
       ROUND(100*SUM(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1 ELSE 0 END)/COUNT(*), 1) as late_percent
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
HAVING COUNT(*) >= 100
ORDER BY avg_days DESC;
""")

Отсекаем штаты с малым числом заказом, так как по ним среднее ненадёжно

In [ ]:
q("""
SELECT COALESCE(t.product_category_name, t.product_category_name_english) AS category,
       COUNT(DISTINCT r.order_id) AS orders,
       ROUND(AVG(r.review_score), 2) AS avg_score
FROM order_reviews r
JOIN order_items oi ON r.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation t
       ON p.product_category_name = t.product_category_name
GROUP BY category
HAVING COUNT(DISTINCT r.order_id) >=100
ORDER BY avg_score
LIMIT 10;
""")

# Блок 3. Клиенты, повторные покупки, когорты, RFM

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

## Графики

In [ ]:
rev = q("""
SELECT strftime('%Y-%m', orders.order_purchase_timestamp) AS month,
       COUNT(DISTINCT orders.order_id) AS orders,
       ROUND(SUM(order_items.price),2) AS revenue
FROM orders
JOIN order_items ON orders.order_id = order_items.order_id
WHERE orders.order_status = 'delivered'
GROUP BY month
ORDER BY month;
""")
rev = rev[(rev["month"] >= "2017-01") & (rev["month"] <= "2018-08")]

plt.figure(figsize=(10,5))
plt.plot(rev["month"], rev["revenue"], marker = "o")
plt.xticks(rotation = 60)
plt.title = ("Выручка по месяцам (доставленные заказы)")
plt.ylabel("Выручка")
plt.tight_layout()
plt.savefig("../reports//figures/revenue_by_month.png", dpi=150)
plt.show()

In [ ]:
delay = q("""
SELECT CASE 
         WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <=0 THEN '0. Вовремя'
         WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <=3 THEN '1. До 3 дней'
         WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) <=7 THEN '2. До 7 дней'
         ELSE '3. Больше 7 дней'
       END AS delay_bucket,
       COUNT(*) AS reviews,
       ROUND(AVG(r.review_score), 2) AS avg_score
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
GROUP BY delay_bucket
ORDER BY delay_bucket;
""")
plt.figure(figsize=(8, 5))
sns.barplot(data = delay, x = "delay_bucket", y = "avg_score", color ="#4C72B0")
plt.ylim(1, 5)
plt.title = ("Средняя оценка в зависимости от опоздания доставки")
plt.xlabel("Опоздание относительно обещанной даты")
plt.ylabel("Средняя оценка")
plt.tight_layout()
plt.savefig("../reports/figures/score_by_delay.png", dpi = 150)
plt.show()